In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import json



In [39]:
df = pd.read_excel(r'C:\Users\Maxim.Olshansky\PycharmProjects\ndv_parcing\OpenAI\СтадииLast.xlsx')

In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID дом.рф          168 non-null    int64  
 1   Стадия             167 non-null    str    
 2   Предыдущая стадия  168 non-null    str    
 3   Этажность          167 non-null    float64
dtypes: float64(1), int64(1), str(2)
memory usage: 5.4 KB


In [15]:
df["Стадия"] = df["Стадия"].apply(json.loads)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [6]:
print(type(df.loc[0, "Стадия"]))

<class 'str'>


In [28]:
df.loc[1, "Стадия"]["windows_percent"]

0

In [16]:
for i, value in enumerate(df["Стадия"]):
    try:
        json.loads(value)
    except Exception as e:
        print(f"Ошибка в строке {i}: {e}")
        print(repr(value))
        break

Ошибка в строке 6: Expecting value: line 1 column 1 (char 0)
'```json\n{\n  "estimated_floors": 0,\n  "total_floors": 19,\n  "height_percent": 0,\n  "pit_visible": true,\n  "foundation_visible": true,\n  "facade_percent": 30,\n  "windows_percent": 40,\n  "finishing_percent": 25,\n  "landscaping_percent": 0,\n  "confidence": 85\n}\n```'


In [41]:
def parse_stage(value):
    if pd.isna(value):
        return None

    value = value.strip()

    # Убираем Markdown-блок
    if value.startswith("```"):
        lines = value.splitlines()

        # Убираем первую строку ```json
        if lines[0].startswith("```"):
            lines = lines[1:]

        # Убираем последнюю строку ```
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]

        value = "\n".join(lines)

    return json.loads(value)

In [42]:
df["Стадия"] = df["Стадия"].apply(parse_stage)

In [43]:
def get_construction_stage(stage):
    """
    Определяет стадию строительства по данным анализа изображения.

    Parameters
    ----------
    stage : dict
        Словарь с полями:
        - height_percent
        - pit_visible
        - foundation_visible
        - facade_percent
        - windows_percent

    Returns
    -------
    str
        Название стадии строительства.
    """

    if not isinstance(stage, dict):
        return None

    height = stage.get("height_percent", 0)
    pit = stage.get("pit_visible", False)
    foundation = stage.get("foundation_visible", False)
    facade = stage.get("facade_percent", 0)
    windows = stage.get("windows_percent", 0)


    # Фундамент
    if (
        height < 20

    ):
        return "Начальная стадия"

    # Монтажные работы
    if 20 <= height <= 60:
        return "Монтажные работы"

    # Завершающий цикл
    if height > 60:
        return "Завершающий цикл"

    return "Не удалось определить"

In [44]:
df["Этап"] = df["Стадия"].apply(get_construction_stage)

In [45]:
df.to_excel('NewStage2.xlsx', index=False)